In [ ]:
"""
PHẦN 1: XÂY DỰNG DATASET 1

"""
# ==============================================================================
# --- 0. THIẾT LẬP MÔI TRƯỜNG ---
# ==============================================================================
print("--- Bước 0: Thiết lập môi trường ---")

# Cài đặt thư viện cần thiết
!pip install -q underthesea wordcloud

# Import các thư viện
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import numpy as np
import requests
import re
from collections import Counter
from underthesea import word_tokenize
from wordcloud import WordCloud
import string
import unicodedata
import io


print("✅ Import thư viện thành công!")

# Kết nối Google Colab với Google Drive
print("\nĐang kết nối với Google Drive...")
drive.mount('/content/drive')
print("✅ Kết nối thành công!")

# Thiết lập các đường dẫn
BASE_PATH = '/content/drive/MyDrive/tram_cam'
TRANSLATED_FILE_PATH = os.path.join(BASE_PATH, 'file_dich.csv') # File dịch
REDDIT_CRAWL_PATH = os.path.join(BASE_PATH, 'reddit_depression_crawl.csv') # File crawl mới
FINAL_UNIFIED_DATASET_PATH = os.path.join(BASE_PATH, 'dataset_1.csv') # File output cuối cùng

print(f"\nĐường dẫn làm việc: {BASE_PATH}")


# ==============================================================================
# --- BƯỚC 1. TẢI VÀ CHUẨN BỊ DỮ LIỆU NHÃN 1 (TỪ FILE DỊCH GỐC REDDIT) ---
# ==============================================================================
print("\n--- Bước 1: Chuẩn bị dữ liệu Nhãn 1 ---")
try:
    df_translated_full = pd.read_csv(
        TRANSLATED_FILE_PATH, header=None, names=['id', 'clean_text_en', 'text_vi', 'sentiment']
    )
    df_label_1_raw = df_translated_full[df_translated_full['sentiment'] == '1'].copy()

    df_label_1 = pd.DataFrame()
    df_label_1['text_vi'] = df_label_1_raw['text_vi'] # Đặt tên cột là 'text_vi' cho thống nhất
    df_label_1['sentiment'] = 1
    df_label_1.dropna(subset=['text_vi'], inplace=True)

    print(f"✅ Đã chuẩn bị thành công {len(df_label_1)} mẫu cho Nhãn 1.")

except Exception as e:
    print(f"⚠️ Lỗi khi xử lý file dịch: {e}")

# ==============================================================================
# --- BƯỚC 2. TẢI VÀ CHUẨN BỊ DỮ LIỆU NHÃN 0 (TỪ FILE REDDIT CRAWL) ---
# ==============================================================================
print("\n--- Bước 2: Chuẩn bị dữ liệu Nhãn 0 ---")
try:
    df_crawl_full = pd.read_csv(REDDIT_CRAWL_PATH)

    # Lọc ra các dòng có nhãn là 0
    # Cần đảm bảo cột sentiment là kiểu số để so sánh
    df_crawl_full['sentiment'] = pd.to_numeric(df_crawl_full['sentiment'], errors='coerce')
    df_label_0_raw = df_crawl_full[df_crawl_full['sentiment'] == 0].copy()

    # Tạo DataFrame chuẩn cho Nhãn 0
    df_label_0 = pd.DataFrame()
    df_label_0['text_vi'] = df_label_0_raw['text'] # Lấy cột 'text'
    df_label_0['sentiment'] = 0
    df_label_0.dropna(subset=['text_vi'], inplace=True)

    print(f"✅ Đã chuẩn bị thành công {len(df_label_0)} mẫu cho Nhãn 0 từ file crawl.")
except FileNotFoundError:
    print(f"⚠️ Lỗi: Không tìm thấy file tại '{REDDIT_CRAWL_PATH}'. Vui lòng kiểm tra lại.")
except Exception as e:
    print(f"⚠️ Lỗi khi xử lý file crawl: {e}")

# ==============================================================================
# --- BƯỚC 3. GỘP, XÁO TRỘN VÀ LƯU BỘ DỮ LIỆU CUỐI CÙNG ---
# ==============================================================================
if 'df_label_1' in locals() and 'df_label_0' in locals():
    print("\n--- Bước 3: Gộp, xáo trộn và lưu kết quả ---")

    # Gộp hai DataFrame lại
    df_final = pd.concat([df_label_1, df_label_0], ignore_index=True)

    # Xóa các dòng trùng lặp lần cuối
    df_final.drop_duplicates(subset=['text_vi'], inplace=True)

    # Xáo trộn toàn bộ DataFrame một cách ngẫu nhiên
    df_final_shuffled = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"Đã gộp 2 bộ dữ liệu. Tổng số dòng cuối cùng: {len(df_final_shuffled)}")
    print("Phân bổ nhãn trong dữ liệu mới:")
    print(df_final_shuffled['sentiment'].value_counts())

    # Lưu bộ dữ liệu cuối cùng ra file CSV
    df_final_shuffled.to_csv(FINAL_UNIFIED_DATASET_PATH, index=False, encoding='utf-8-sig')

    print(f"\n✅ HOÀN TẤT! Đã lưu bộ dữ liệu thống nhất cuối cùng tại: {FINAL_UNIFIED_DATASET_PATH}")
    print("Xem trước 5 dòng đầu của bộ dữ liệu cuối cùng:")
    display(df_final_shuffled.head())

else:
    print("\n⚠️ Không thể hoàn tất do thiếu dữ liệu Nhãn 0 hoặc Nhãn 1.")

--- Bước 0: Thiết lập môi trường ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.2 MB/s eta 0:00:00
✅ Import thư viện thành công!

Đang kết nối với Google Drive...
Mounted at /content/drive
✅ Kết nối thành công!

Đường dẫn làm việc: /content/drive/MyDrive/tram_cam

--- Bước 1: Chuẩn bị dữ liệu Nhãn 1 ---
✅ Đã chuẩn bị thành công 3831 mẫu cho Nhãn 1.

--- Bước 2: Chuẩn bị dữ liệu Nhãn 0 ---
✅ Đã chuẩn bị thành công 2151 mẫu cho Nhãn 0 từ file crawl.

--- Bước 3: Gộp, xáo trộn và lưu kết quả ---
Đã gộp 2 bộ dữ liệu. Tổng số dòng cuối cùng: 5933
Phân bổ nhãn trong dữ liệu mới:
sentiment
1    3782
0    2151
Name: count, dtype: int64

✅ HOÀN TẤT! Đã lưu bộ dữ liệu thống nhất cuối cùng tại: /content/drive/MyDrive/tram_cam/dataset_1.csv
Xem trước 5 dòng đầu của bộ dữ liệu cuối cùng:


,text_vi,sentiment
0,Tại sao tôi lại dễ bị nóng và lo âu trở nên tồ...,1
1,Tôi bị rối loạn lo âu chung. Tôi đã gặp một vụ...,1
2,"Người ta nói rằng Chúa chỉ thử thách chúng ta,...",1
3,"imohumoren, đó chính là chỗ tôi gặp vấn đề — k...",1
4,"Hôm qua tôi lại phải đến bác sĩ, đơn thuốc của...",1


In [ ]:
"""
PHẦN 1: XÂY DỰNG DATASET 2

"""

# ==============================================================================
# --- 0. THIẾT LẬP MÔI TRƯỜNG ---
# ==============================================================================
print("--- Bước 0: Thiết lập môi trường ---")

# Cài đặt thư viện cần thiết
!pip install -q underthesea wordcloud

# Import các thư viện
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import numpy as np
import requests
import re
from collections import Counter
from underthesea import word_tokenize
from wordcloud import WordCloud
import string

print("✅ Import thư viện thành công!")

# Kết nối Google Colab với Google Drive
print("\nĐang kết nối với Google Drive...")
drive.mount('/content/drive')
print("✅ Kết nối thành công!")

# Thiết lập các đường dẫn
BASE_PATH = '/content/drive/MyDrive/tram_cam'
TRANSLATED_FILE_PATH = os.path.join(BASE_PATH, 'file_dich.csv')
VSMEC_PATH = os.path.join(BASE_PATH, 'UIT-VSMEC')
FINAL_DATASET_PATH = os.path.join(BASE_PATH, 'dataset_2.csv')

print(f"\nĐường dẫn làm việc: {BASE_PATH}")


# ##############################################################################
# #### PHẦN 1: XÂY DỰNG BỘ DỮ LIỆU THÔ                                     ####
# ##############################################################################

# ==============================================================================
# --- 1. TẢI VÀ CHUẨN BỊ DỮ LIỆU NHÃN 1 (CÓ DẤU HIỆU TRẦM CẢM) ---
# ==============================================================================
print("\n--- Bước 1: Chuẩn bị dữ liệu Nhãn 1 ---")
try:
    df_translated_full = pd.read_csv(
        TRANSLATED_FILE_PATH, header=None, names=['id', 'clean_text_en', 'text_vi', 'sentiment']
    )
    df_label_1_raw = df_translated_full[df_translated_full['sentiment'] == '1'].copy()
    df_label_1 = pd.DataFrame()
    df_label_1['text_vi'] = df_label_1_raw['text_vi']
    df_label_1['sentiment'] = 1
    df_label_1.dropna(subset=['text_vi'], inplace=True)
    print(f"✅ Đã chuẩn bị thành công {len(df_label_1)} mẫu cho Nhãn 1.")
except Exception as e:
    print(f"⚠️ Lỗi khi xử lý file dịch: {e}")

# ==============================================================================
# --- 2. TẢI VÀ CHUẨN BỊ DỮ LIỆU NHÃN 0 (KHÔNG CÓ DẤU HIỆU) ---
# ==============================================================================
print("\n--- Bước 2: Chuẩn bị dữ liệu Nhãn 0 ---")
files_to_read = ['train_nor_811.xlsx', 'valid_nor_811.xlsx', 'test_nor_811.xlsx']
column_names = ['original_id', 'Emotion', 'Sentence']
list_of_dfs = []
for file_name in files_to_read:
    file_path = os.path.join(VSMEC_PATH, file_name)
    try:
        df_temp = pd.read_excel(file_path, header=None, names=column_names)
        list_of_dfs.append(df_temp)
    except Exception as e:
        print(f"⚠️ Lỗi khi đọc file {file_name}: {e}")

if list_of_dfs:
    df_vsmec_raw = pd.concat(list_of_dfs, ignore_index=True)
    df_vsmec_clean = df_vsmec_raw[df_vsmec_raw['Emotion'] != 'Emotion'].copy()
    emotions_to_keep = ['Enjoyment', 'Surprise', 'Other']
    df_label_0_source = df_vsmec_clean[df_vsmec_clean['Emotion'].isin(emotions_to_keep)].copy()
    df_label_0 = pd.DataFrame()
    df_label_0['text_vi'] = df_label_0_source['Sentence']
    df_label_0['sentiment'] = 0
    df_label_0.dropna(subset=['text_vi'], inplace=True)
    print(f"✅ Đã chuẩn bị thành công {len(df_label_0)} mẫu cho Nhãn 0 từ VSMEC.")
else:
    print("⚠️ Không đọc được dữ liệu từ VSMEC.")

# ==============================================================================
# --- 3. GỘP, XÁO TRỘN VÀ LƯU BỘ DỮ LIỆU THÔ ---
# ==============================================================================
if 'df_label_1' in locals() and 'df_label_0' in locals():
    print("\n--- Bước 3: Gộp và xáo trộn ---")
    df_final = pd.concat([df_label_1, df_label_0], ignore_index=True)
    df_final.drop_duplicates(subset=['text_vi'], inplace=True)
    df_final_shuffled = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    df_final_shuffled.to_csv(FINAL_DATASET_PATH, index=False, encoding='utf-8-sig')
    print(f"✅ HOÀN TẤT PHẦN 1! Đã lưu bộ dữ liệu thô tại: {FINAL_DATASET_PATH} ({len(df_final_shuffled)} dòng)")
    df = df_final_shuffled.copy()
else:
    print("\n⚠️ Không thể hoàn tất do thiếu dữ liệu Nhãn 0 hoặc Nhãn 1.")
    df = None



--- Bước 0: Thiết lập môi trường ---
✅ Import thư viện thành công!

Đang kết nối với Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Kết nối thành công!

Đường dẫn làm việc: /content/drive/MyDrive/tram_cam

--- Bước 1: Chuẩn bị dữ liệu Nhãn 1 ---
✅ Đã chuẩn bị thành công 3831 mẫu cho Nhãn 1.

--- Bước 2: Chuẩn bị dữ liệu Nhãn 0 ---
✅ Đã chuẩn bị thành công 3565 mẫu cho Nhãn 0 từ VSMEC.

--- Bước 3: Gộp và xáo trộn ---
✅ HOÀN TẤT PHẦN 1! Đã lưu bộ dữ liệu thô tại: /content/drive/MyDrive/tram_cam/dataset_2.csv (7345 dòng)
